# Graph RAG

BM25 and dense retrieval both compare the question to each passage on its own. If the passage that answers the question does not share words, or an embedding, with the question, it is never found.

Graph RAG reads the corpus first. A model goes through every passage once and writes down the facts it states as triples: a subject, a relation, and an object. "Google Services generates revenues from advertising" becomes `(Google Services, generates revenues from, advertising)`. Every subject and object becomes a node in a graph and every relation becomes an edge between them. Each node remembers which passages mentioned it.

At query time the entities the question names are looked up in the graph, and the retriever walks a couple of hops out from them. The passages that mention the entities it reached come back. A passage can be reached this way even when it repeats none of the words in the question, because the connection was made when the graph was built, not when the question was asked.

## Load in documents and chunk them

The chunks are exactly the ones the dense and HyDE notebooks use, so the pages each retriever returns can be compared at the end.

In [1]:
from rag.documents import load_documents
from rag.chunk import chunk_documents

FILE_PATH = "/home/nick/github-projects/Sec-Rag/data/google_10K.pdf"

documents = load_documents(FILE_PATH)

chunks = chunk_documents(
    documents=documents,
    chunk_size=300,
    chunk_overlap=30
)

print(f"Length of Documents: {len(documents)}")
print(f"Length of Chunks: {len(chunks)}")

/home/nick/github-projects/Sec-Rag/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length of Documents: 107
Length of Chunks: 1437


## Create the generator

The graph is built by a chat model, not by an embedding model. The generator is the same `LangChainGenerator` the HyDE notebook uses, with `gpt-4o-mini` behind it and `OPENAI_API_KEY` read from `.env` by `load_dotenv()`.

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from rag.llm import DEFAULT_CHAT_MODEL, LangChainGenerator

load_dotenv()

generator = LangChainGenerator(
    ChatOpenAI(model=DEFAULT_CHAT_MODEL, max_tokens=256)
    )

## Look at the triples from one chunk

This is the prompt every chunk is sent with, followed by what comes back for one chunk from the Business section. `extract_triples` parses the model's JSON reply into `Triple` objects and returns an empty list if the reply cannot be parsed, so one bad reply never stops the rest of the corpus from being indexed.

In [4]:
from rag.graph import extract_triples
from rag.prompts import TRIPLE_PROMPT

print(TRIPLE_PROMPT)

chunk = chunks[50]
print(f"\n[page {chunk.metadata['page']}]\n{chunk.page_content[:400]}\n")

for triple in extract_triples(chunk.page_content, generator):
    print(f"({triple.subject}) -[{triple.relation}]-> ({triple.object})")

Extract the factual relationships stated in the passage below from a SEC 10-K filing. Reply with a JSON list of objects, each holding exactly the keys "subject", "relation" and "object". Use the wording of the passage. Reply with an empty list if it states no relationships.

Passage:
{text}

JSON:

[page 3]
small. Our mission to organize the world’s information and make it universally accessible and useful is as relevant today
as it was when we were founded in 1998. Since then, we have evolved from a company that helps people find answers
to a company that also helps people get things done.

(Our mission) -[is]-> (to organize the world’s information and make it universally accessible and useful)
(We) -[were founded in]-> (1998)
(We) -[have evolved from]-> (a company that helps people find answers to a company that also helps people get things done)


## Resolve entity aliases

A filing writes the same company as "Alphabet Inc.", "Alphabet Inc" and "Alphabet" on consecutive pages. Keyed literally those are three separate nodes and a walk that should cross between them stops short.

`EntityResolver` lowercases each mention and embeds it with the bi-encoder. A mention whose cosine similarity to an existing key is at or above the threshold is mapped onto that key instead of becoming a new one. Different companies stay apart.

In [5]:
from rag.graph import EntityResolver

resolver = EntityResolver()

for mention in ["Alphabet Inc.", "Alphabet Inc", "Alphabet", "Google", "YouTube", "Google Cloud"]:
    print(f"{mention!r:18} -> {resolver.resolve(mention)!r}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6351.27it/s]


'Alphabet Inc.'    -> 'alphabet inc.'
'Alphabet Inc'     -> 'alphabet inc.'
'Alphabet'         -> 'alphabet'
'Google'           -> 'google'
'YouTube'          -> 'youtube'
'Google Cloud'     -> 'google cloud'


## Build the graph

`GraphRetriever` runs `extract_triples` on every chunk, resolves each subject and object through the resolver, and adds an edge between them carrying the relation. That is one model call per chunk, so this cell takes a while for 295 chunks. The graph is undirected: "Alphabet designs TPUs" should be reachable from TPUs just as much as from Alphabet.

In [6]:
from rag.graph import GraphRetriever

retriever = GraphRetriever(generator, hops=3)
retriever.add_documents(chunks)

print(f"Nodes: {retriever.graph.number_of_nodes()}")
print(f"Edges: {retriever.graph.number_of_edges()}\n")

for subject, obj, relation in list(retriever.graph.edges(data="relation"))[:10]:
    print(f"({subject}) -[{relation}]-> ({obj})")

Nodes: 1213
Edges: 1228

(class a common stock) -[has par value]-> ($0.001)
(class a common stock) -[is traded on]-> (nasdaq stock market llc (nasdaq global select market))
(class a common stock) -[is entitled to]-> (10 votes per share)
(class a common stock) -[represented approximately]-> (52.7% of the voting power of our outstanding common stock)
(class a common stock) -[has been listed on]-> (the nasdaq global select market under the symbol 'goog' since august 19, 2004)
(class a common stock) -[is neither listed nor traded]-> ()
(class a common stock) -[may be converted at any time at the option of]-> (stockholders)
(class a common stock) -[automatically convert upon sale or transfer to]-> (class a common stock)
(class a common stock) -[to sell up to]-> (the trading plan)
($0.001) -[has par value]-> (class c capital stock)


## Retrieve for a query

The query is tokenized and any graph entity that appears in it as a whole, contiguous run of words is a starting point. From each starting point the retriever collects every node within `hops` edges. Passages are then ranked by how many of those reached entities they mention, so a passage that covers several of them outranks one that repeats a single name.

In [7]:
query = "how much did Google spend on research and development in 2025"

results = retriever.retrieve(query, top_k=4)

for doc in results:
    print(f"\n[page {doc.metadata['page']}]\n{doc.page_content[:200]}")


[page 60]
Table of Contents Alphabet Inc.
• consumer subscriptions, which primarily include revenues from YouTube services, such as YouTube TV,
YouTube Music and Premium, and NFL Sunday Ticket, as well as Googl

[page 61]
Table of Contents Alphabet Inc.
◦ content acquisition costs, which are payments to content providers from whom we license video and
other content for distribution, primarily related to YouTube (we pay

[page 5]
• consumer subscriptions, which primarily include revenues from YouTube services, such as YouTube TV,
YouTube Music and Premium, and NFL Sunday Ticket, as well as Google One, which offers access to ou

[page 58]
Table of Contents Alphabet Inc.
Alphabet Inc.
CONSOLIDATED STATEMENTS OF CASH FLOWS
(in millions)
  Year Ended December 31,
  2023 2024 2025
Operating activities
Net income $ 73,795  $ 100,118  $ 132,
